# Data Transforms Pipeline

Standalone data transforms for volatility forecasting:  
diurnal adjustment, semantic transforms, rolling winsorization,
and the combined `robust_transform` pipeline.

In [ ]:
import sys

sys.path.insert(0, "/content/harxhar/colab")

from src.loading import load_raw_data

data = load_raw_data("all30min")
print(f"Loaded {len(data)} rows, {len(data.columns)} columns")
data.head()

In [ ]:
# export
"""Standalone data transforms and feature generation for volatility forecasting."""

from __future__ import annotations

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

PERIODS_PER_DAY: int = 48
DIURNAL_WINDOW: int = 20
SEGMENT_CHOICES: list[str] = ["all", "morning", "midday", "closing", "overnight"]
DIURNAL_MIN_PERIODS: int = 5
WINSOR_LOWER_Q: float = 0.05
WINSOR_UPPER_Q: float = 0.95
SKIP_VARS: set[str] = {"hour", "DOW", "t", "date"}
DIURNAL_EXCLUDED: set[str] = SKIP_VARS | {"vix", "sentiment"}


def _hhmm(h: int, m: int) -> int:
    return h * 60 + m


SEGMENT_DEFINITIONS: dict[str, tuple[int, int]] = {
    "morning": (_hhmm(8, 30), _hhmm(11, 0)),
    "midday": (_hhmm(10, 30), _hhmm(14, 30)),
    "closing": (_hhmm(14, 0), _hhmm(16, 0)),
    "overnight": (_hhmm(16, 30), _hhmm(8, 30)),
}


def slice_to_segment(df: pd.DataFrame, segment: str) -> pd.DataFrame:
    """Filter rows to a time-of-day segment. Handles midnight wrap-around."""
    start, end = SEGMENT_DEFINITIONS[segment]
    minutes = df["t"].dt.hour * 60 + df["t"].dt.minute
    if start < end:
        mask = (minutes >= start) & (minutes <= end)
    else:
        mask = (minutes >= start) | (minutes <= end)
    return df.loc[mask].reset_index(drop=True)


def compute_segment_train_window(dates: pd.Series, train_window_days: int) -> int:
    """Compute train window in periods using median slots/day for a segment."""
    daily_counts = pd.Series(dates).dt.date.value_counts()
    median_slots = int(daily_counts.median())
    return train_window_days * median_slots


In [ ]:
# Test constants and slice_to_segment (backfilled - not previously exercised)
print("SEGMENT_CHOICES:", SEGMENT_CHOICES)
print("SEGMENT_DEFINITIONS:", SEGMENT_DEFINITIONS)

# slice_to_segment needs a column named "t" with a datetime dtype
_seg_df = pd.DataFrame({"t": pd.to_datetime(pd.Series([
    "2023-01-03 09:00", "2023-01-03 10:30", "2023-01-03 13:00",
    "2023-01-03 15:30", "2023-01-03 18:00", "2023-01-03 02:00",
]))})
for seg in ("morning", "midday", "closing", "overnight"):
    sl = slice_to_segment(_seg_df, seg)
    print(f"{seg}: {len(sl)} rows -> {sl['t'].tolist()}")

# compute_segment_train_window smoke test
print("train window (3 days, 2 slots/day demo):",
      compute_segment_train_window(_seg_df["t"], train_window_days=3))


In [ ]:
# export
# ---------------------------------------------------------------------------
# Diurnal adjustment
# ---------------------------------------------------------------------------


def diurnal_adjust(
    series: pd.Series,
    time_of_day_series: pd.Series,
    has_negatives: bool,
    window: int = DIURNAL_WINDOW,
    min_periods: int = DIURNAL_MIN_PERIODS,
) -> tuple[pd.Series, pd.Series]:
    """Remove intraday seasonality via rolling per-slot baseline.

    Causality (audit-relevant):
    Within each time-of-day slot, the rolling statistic is followed by
    ``.shift(1)`` so that the baseline at time *t* is computed from the
    ``window`` most recent in-slot observations strictly before *t*. The
    adjusted value ``series[t] / baseline[t]`` therefore uses no information
    from time *t* itself or later. This is part of the project-wide strict-
    causality invariant (see also ``generate_har_features`` and
    ``rolling_winsorize``); any forecast produced downstream is guaranteed
    free of look-ahead bias from the diurnal stage.

    Parameters
    ----------
    series : pd.Series
        Raw values to adjust.
    time_of_day_series : pd.Series
        Aligned series of time-of-day slot labels (same index as *series*).
    has_negatives : bool
        If True the variable can be negative and the baseline is rolling std;
        otherwise the baseline is rolling mean.
    window, min_periods : int
        Rolling window parameters applied *within* each slot. NaN baselines
        (warm-up rows lacking ``min_periods`` history) are filled with 1.0
        so that the adjusted value equals the raw value during warm-up.

    Returns
    -------
    (adjusted, baseline) where adjusted = series / baseline.
    """
    df = pd.DataFrame({"val": series, "slot": time_of_day_series})

    if has_negatives:
        baseline = df.groupby("slot")["val"].transform(
            lambda g: g.rolling(window, min_periods=min_periods).std().shift(1)
        )
    else:
        baseline = df.groupby("slot")["val"].transform(
            lambda g: g.rolling(window, min_periods=min_periods).mean().shift(1)
        )

    baseline = baseline.fillna(1.0)
    adjusted = series / baseline
    return adjusted, baseline


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Diurnal adjustment: remove intraday seasonality via rolling per-slot baseline ---
DIURNAL_WINDOW = 20
DIURNAL_MIN_PERIODS = 5

rv = data["RV"].copy()
tod = data["time_of_day"].copy()
has_neg = bool((rv.dropna() < 0).any())

# Build per-slot rolling baseline (mean for non-negative, std for negative)
df_tmp = pd.DataFrame({"val": rv, "slot": tod})
if has_neg:
    rv_baseline = df_tmp.groupby("slot")["val"].transform(
        lambda g: g.rolling(DIURNAL_WINDOW, min_periods=DIURNAL_MIN_PERIODS).std().shift(1)
    )
else:
    rv_baseline = df_tmp.groupby("slot")["val"].transform(
        lambda g: g.rolling(DIURNAL_WINDOW, min_periods=DIURNAL_MIN_PERIODS).mean().shift(1)
    )
rv_baseline = rv_baseline.fillna(1.0)
rv_adj = rv / rv_baseline

# Pick a short window to visualise
n_show = 13 * 5  # ~5 trading days of 30-min bars
idx = slice(DIURNAL_WINDOW * 13, DIURNAL_WINDOW * 13 + n_show)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(rv.iloc[idx].values, label="raw RV")
axes[0].set_title("Raw RV")
axes[0].legend()

axes[1].plot(rv_baseline.iloc[idx].values, label="baseline", color="orange")
axes[1].set_title("Diurnal baseline (rolling mean per slot, shift=1)")
axes[1].legend()

axes[2].plot(rv_adj.iloc[idx].values, label="adjusted RV", color="green")
axes[2].set_title("Diurnal-adjusted RV")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Baseline stats:\n{rv_baseline.describe()}")

In [ ]:
# export
# ---------------------------------------------------------------------------
# Semantic (column-name-based) transforms
# ---------------------------------------------------------------------------


def apply_semantic_transform(
    series: pd.Series,
    col_name: str,
    has_negatives: bool,
    allow_missing: bool = False,
) -> pd.Series:
    """Apply a variance-stabilising transform chosen by column name.

    Rules (checked in order):
    1. name contains ret2 / RV / turnover / bipow / effspread -> sqrt
    2. name contains autocov -> sign(x) * sqrt(|x|)
    3. name contains ret3 -> cbrt
    4. name contains ret4 -> fourth root (x ** 0.25)
    5. has_negatives or name contains sumabsret -> identity (NaN -> 0)
    6. default -> log
    """
    name = col_name.lower()

    if any(tok in name for tok in ("ret2", "rv", "turnover", "bipow", "effspread")):
        return np.sqrt(series)

    if "autocov" in name:
        return np.sign(series) * np.sqrt(np.abs(series))

    if "ret3" in name:
        return np.cbrt(series)

    if "ret4" in name:
        return np.power(np.abs(series), 0.25) * np.sign(series)

    if has_negatives or "sumabsret" in name:
        out = series.copy()
        if not allow_missing:
            out = out.fillna(0.0)
        return out

    # default: log (guard against non-positive values)
    return np.log(series.clip(lower=1e-12))


In [ ]:
# --- Semantic transforms: before / after histograms ---
examples = {
    "RV": ("sqrt", False),
    "autocov": ("sign*sqrt|x|", True),
}

fig, axes = plt.subplots(len(examples), 2, figsize=(12, 4 * len(examples)))

for i, (col, (label, neg)) in enumerate(examples.items()):
    raw = data[col].dropna()
    name = col.lower()

    # Inline semantic transform logic
    if any(tok in name for tok in ("ret2", "rv", "turnover", "bipow", "effspread")):
        transformed = np.sqrt(data[col]).dropna()
    elif "autocov" in name:
        transformed = (np.sign(data[col]) * np.sqrt(np.abs(data[col]))).dropna()
    elif "ret3" in name:
        transformed = np.cbrt(data[col]).dropna()
    elif "ret4" in name:
        transformed = (np.power(np.abs(data[col]), 0.25) * np.sign(data[col])).dropna()
    elif neg or "sumabsret" in name:
        transformed = data[col].fillna(0.0).dropna()
    else:
        transformed = np.log(data[col].clip(lower=1e-12)).dropna()

    axes[i, 0].hist(raw, bins=80, alpha=0.7)
    axes[i, 0].set_title(f"{col} — raw")

    axes[i, 1].hist(transformed, bins=80, alpha=0.7, color="orange")
    axes[i, 1].set_title(f"{col} — {label}")

plt.tight_layout()
plt.show()

In [ ]:
# export
# ---------------------------------------------------------------------------
# Rolling winsorization
# ---------------------------------------------------------------------------


def rolling_winsorize(
    series: pd.Series,
    window: int = 240,
    allow_missing: bool = False,
    is_target: bool = False,
) -> pd.Series:
    """Clip values to rolling 5th / 95th quantile bounds (shifted by 1).

    Parameters
    ----------
    series : pd.Series
    window : int
        Lookback window for quantile estimation.
    allow_missing : bool
        If True and not is_target, use nanquantile-style (min_periods=1).
    is_target : bool
        Targets never use nanquantile even when allow_missing is True.
    """
    use_nan = allow_missing and not is_target
    min_per = 1 if use_nan else window

    lower = series.rolling(window, min_periods=min_per).quantile(WINSOR_LOWER_Q).shift(1)
    upper = series.rolling(window, min_periods=min_per).quantile(WINSOR_UPPER_Q).shift(1)
    return series.clip(lower=lower, upper=upper)


In [ ]:
# --- Rolling winsorization ---
WINSOR_LOWER_Q = 0.05
WINSOR_UPPER_Q = 0.95

rv_sqrt = np.sqrt(rv)

# Inline: rolling quantile(0.05/0.95).shift(1), then clip
lower = rv_sqrt.rolling(240, min_periods=240).quantile(WINSOR_LOWER_Q).shift(1)
upper = rv_sqrt.rolling(240, min_periods=240).quantile(WINSOR_UPPER_Q).shift(1)
rv_winsor = rv_sqrt.clip(lower=lower, upper=upper)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(rv_sqrt.dropna(), bins=100, alpha=0.7)
axes[0].set_title("sqrt(RV) — before winsorization")

axes[1].hist(rv_winsor.dropna(), bins=100, alpha=0.7, color="orange")
axes[1].set_title("sqrt(RV) — after rolling winsorization")

plt.tight_layout()
plt.show()

# Show how many values were clipped
clipped = (rv_winsor != rv_sqrt).sum()
print(f"Clipped {clipped} / {len(rv_sqrt)} values ({100 * clipped / len(rv_sqrt):.2f}%)")

In [ ]:
# export
# ---------------------------------------------------------------------------
# Full pipeline
# ---------------------------------------------------------------------------


def robust_transform(
    df: pd.DataFrame,
    col_name: str,
    time_col: str = "time_of_day",
    use_transform: bool = True,
    use_diurnal: bool = True,
    allow_missing: bool = False,
    winsor_window: int | None = None,
    is_target: bool = False,
) -> tuple[pd.Series, pd.Series]:
    """Chain diurnal_adjust -> apply_semantic_transform -> rolling_winsorize.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain *col_name* and (if diurnal is used) *time_col*.
    col_name : str
        Column to transform.
    time_col : str
        Column holding the time-of-day slot labels.
    use_transform, use_diurnal : bool
        Toggle individual stages.
    allow_missing : bool
        Forwarded to downstream helpers.
    winsor_window : int | None
        Override default winsorization window (240).
    is_target : bool
        Forwarded to rolling_winsorize.

    Returns
    -------
    (adjusted_series, baseline)
    """
    if col_name in SKIP_VARS:
        return df[col_name].copy(), pd.Series(1.0, index=df.index)

    series = df[col_name].copy()
    has_negatives = bool((series.dropna() < 0).any())

    # --- diurnal ---
    baseline = pd.Series(1.0, index=df.index)
    if use_diurnal and col_name not in DIURNAL_EXCLUDED and time_col in df.columns:
        series, baseline = diurnal_adjust(series, df[time_col], has_negatives)

    # --- semantic transform ---
    if use_transform:
        series = apply_semantic_transform(series, col_name, has_negatives, allow_missing=allow_missing)

    # --- winsorize ---
    ww = winsor_window if winsor_window is not None else 240
    series = rolling_winsorize(series, window=ww, allow_missing=allow_missing, is_target=is_target)

    return series, baseline


In [ ]:
# --- Full robust_transform pipeline on RV (inline 3-step: diurnal -> semantic -> winsorize) ---
SKIP_VARS = {"hour", "DOW", "t", "date"}
DIURNAL_EXCLUDED = SKIP_VARS | {"vix", "sentiment"}

col_name = "RV"
series = data[col_name].copy()
has_negatives = bool((series.dropna() < 0).any())

# Step 1: diurnal adjustment
baseline = pd.Series(1.0, index=data.index)
if col_name not in DIURNAL_EXCLUDED and "time_of_day" in data.columns:
    df_tmp2 = pd.DataFrame({"val": series, "slot": data["time_of_day"]})
    if has_negatives:
        baseline = df_tmp2.groupby("slot")["val"].transform(
            lambda g: g.rolling(DIURNAL_WINDOW, min_periods=DIURNAL_MIN_PERIODS).std().shift(1)
        )
    else:
        baseline = df_tmp2.groupby("slot")["val"].transform(
            lambda g: g.rolling(DIURNAL_WINDOW, min_periods=DIURNAL_MIN_PERIODS).mean().shift(1)
        )
    baseline = baseline.fillna(1.0)
    series = series / baseline

# Step 2: semantic transform (RV -> sqrt)
name_lower = col_name.lower()
if any(tok in name_lower for tok in ("ret2", "rv", "turnover", "bipow", "effspread")):
    series = np.sqrt(series)
elif "autocov" in name_lower:
    series = np.sign(series) * np.sqrt(np.abs(series))
elif "ret3" in name_lower:
    series = np.cbrt(series)
elif "ret4" in name_lower:
    series = np.power(np.abs(series), 0.25) * np.sign(series)
elif has_negatives or "sumabsret" in name_lower:
    series = series.fillna(0.0)
else:
    series = np.log(series.clip(lower=1e-12))

# Step 3: rolling winsorize
lower_b = series.rolling(240, min_periods=240).quantile(WINSOR_LOWER_Q).shift(1)
upper_b = series.rolling(240, min_periods=240).quantile(WINSOR_UPPER_Q).shift(1)
adj_rv = series.clip(lower=lower_b, upper=upper_b)

# Store in DataFrame for downstream cells
data["adj_RV"] = adj_rv

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

show = slice(0, 13 * 20)  # ~20 days

axes[0].plot(data["RV"].iloc[show].values, label="raw RV", alpha=0.7)
axes[0].set_title("Raw RV")
axes[0].legend()

axes[1].plot(adj_rv.iloc[show].values, label="adj_RV (robust_transform)", color="green", alpha=0.7)
axes[1].set_title("Fully transformed RV")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"adj_RV stats:\n{adj_rv.describe()}")

In [ ]:
# export
# ---------------------------------------------------------------------------
# HAR lag features
# ---------------------------------------------------------------------------


def resolve_har_lags(max_lag: int = 3125) -> list[int]:
    """Powers-of-5 lag sequence: [1, 5, 25, 125, 625, 3125]."""
    seq, v = [], 1
    while v <= max_lag:
        seq.append(v)
        v *= 5
    return seq


def generate_har_features(
    df: pd.DataFrame,
    target_col: str = "adj_RV",
    exog_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, list[str]]:
    """Add rolling-mean HAR features (shifted by 1) for each powers-of-5 lag.

    Features are generated for *target_col* and each column in *exog_cols*.
    Target features are named ``har_ma_{lag}``; exog features ``{col}_ma_{lag}``.
    """
    lags = resolve_har_lags()
    features: dict[str, pd.Series] = {}
    feature_names: list[str] = []
    for col in [target_col] + (exog_cols or []):
        for lag in lags:
            name = f"har_ma_{lag}" if col == target_col else f"{col}_ma_{lag}"
            features[name] = df[col].rolling(window=lag, min_periods=1).mean().shift(1)
            feature_names.append(name)
    feat_df = pd.DataFrame(features, index=df.index)
    return pd.concat([df, feat_df], axis=1), feature_names


In [ ]:
# --- HAR lag features: powers-of-5 rolling means ---
PERIODS_PER_DAY = 48

har_lags, v = [], 1
while v <= 3125:
    har_lags.append(v)
    v *= 5
print(f"HAR lags: {har_lags}")

# Generate rolling mean features
har_names = []
for lag in har_lags:
    name = f"har_ma_{lag}"
    data[name] = data["adj_RV"].rolling(window=lag, min_periods=1).mean().shift(1)
    har_names.append(name)

print(f"Features: {har_names}")
print(data[har_names].iloc[3200:3205])

In [ ]:
# export
# ---------------------------------------------------------------------------
# Calendar features
# ---------------------------------------------------------------------------


def add_calendar_features(df: pd.DataFrame) -> list[str]:
    """Add day-of-week (0-6) and hour features. Returns new column names."""
    df["DOW"] = df["t"].dt.dayofweek
    df["hour"] = df["t"].dt.hour
    return ["DOW", "hour"]


In [ ]:
# --- Calendar features: day-of-week and hour ---
data["DOW"] = data["t"].dt.dayofweek
data["hour"] = data["t"].dt.hour
cal_names = ["DOW", "hour"]
print(f"Calendar features: {cal_names}")
print(data[cal_names].head())

In [ ]:
# export
# ---------------------------------------------------------------------------
# Horizon shift
# ---------------------------------------------------------------------------


def apply_horizon_shift(
    X: np.ndarray,
    y: np.ndarray,
    dates: pd.Series,
    baselines: np.ndarray,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, pd.Series, np.ndarray]:
    """Align features at time *t* with target at *t + horizon*.

    When horizon <= 1 the arrays are returned unchanged.
    """
    if horizon <= 1:
        return X, y, dates, baselines
    shift = horizon - 1
    return (
        X[:-shift],
        y[shift:],
        dates.iloc[:-shift].reset_index(drop=True),
        baselines[shift:],
    )


In [ ]:
# --- Horizon shift: align features at time t with target at t+horizon ---
import numpy as np

# Demo with horizon=2
X_demo = data[har_names].values[:100].astype(np.float64)
y_demo = data["adj_RV"].values[:100].astype(np.float64)
dates_demo = data["t"].iloc[:100]
baselines_demo = baseline.values[:100].astype(np.float64)

horizon = 2
shift = horizon - 1
X_shifted = X_demo[:-shift]
y_shifted = y_demo[shift:]
dates_shifted = dates_demo.iloc[:-shift].reset_index(drop=True)
baselines_shifted = baselines_demo[shift:]
print(f"Before shift: X={X_demo.shape}, y={y_demo.shape}")
print(f"After shift (h={horizon}): X={X_shifted.shape}, y={y_shifted.shape}")

In [ ]:
# export
# ---------------------------------------------------------------------------
# PCA lag features
# ---------------------------------------------------------------------------


def resolve_pca_lags(max_lag: int = 3125, num_points: int = 20) -> list[int]:
    """Generate log-spaced lag indices from 1 to *max_lag*."""
    raw = np.geomspace(1, max_lag, num=num_points)
    return sorted(set(int(round(v)) for v in raw))


def generate_raw_lag_features(
    df: pd.DataFrame,
    target_col: str = "adj_RV",
    max_lag: int = 3125,
    exog_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, list[str]]:
    """Create shifted-lag columns for each log-spaced lag.

    Features are generated for *target_col* and each column in *exog_cols*.
    """
    lags = resolve_pca_lags(max_lag)
    features: dict[str, pd.Series] = {}
    feature_names: list[str] = []
    for col in [target_col] + (exog_cols or []):
        for lag in lags:
            name = f"{col}_lag_{lag}"
            features[name] = df[col].shift(lag)
            feature_names.append(name)
    feat_df = pd.DataFrame(features, index=df.index)
    return pd.concat([df, feat_df], axis=1), feature_names


In [ ]:
# --- PCA lags: log-spaced lag indices ---
pca_lags = sorted(set(int(round(v)) for v in np.geomspace(1, 3125, num=20)))
print(f"PCA lags ({len(pca_lags)}): {pca_lags}")

# Generate shifted-lag columns
pca_names = []
for lag in pca_lags:
    name = f"adj_RV_lag_{lag}"
    data[name] = data["adj_RV"].shift(lag)
    pca_names.append(name)
print(f"PCA feature columns: {len(pca_names)}")

In [ ]:
import sys; sys.path.insert(0, ".."); from _exporter import export_notebook; export_notebook("02_transforms.ipynb", "../../src/transforms.py")
